In [9]:
import nltk
import numpy as np
import pandas as pd
import tensorflow as tf

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout,Bidirectional
from tensorflow.keras.optimizers import Adam

In [10]:
nltk.download("gutenberg")
from nltk.corpus import gutenberg

data=gutenberg.raw('shakespeare-hamlet.txt')
with open('hamlet.txt','w') as file:
    file.write(data)

[nltk_data] Downloading package gutenberg to /root/nltk_data...
[nltk_data]   Package gutenberg is already up-to-date!


In [11]:
with open('hamlet.txt') as file:
  text = file.read().lower()

tokenizer = Tokenizer()
tokenizer.fit_on_texts([text])
total_words=len(tokenizer.word_index) + 1
total_words

4818

In [12]:
input_sequences = []

for line in text.split('\n'):
  token_list = tokenizer.texts_to_sequences([line])[0]
  for i in range(1, len(token_list)):
    n_gram_seq = token_list[:i+1]
    input_sequences.append(n_gram_seq)

max_seq_length = max([len(x) for x in input_sequences])

In [21]:
from tensorflow.keras.preprocessing.sequence import pad_sequences
input_sequences = pad_sequences(input_sequences, maxlen = max_seq_length, padding = 'pre')

In [22]:
x = input_sequences[:, :-1]
y = input_sequences[:, -1]

y = tf.keras.utils.to_categorical(y, num_classes = total_words)

In [30]:
from sklearn.model_selection import train_test_split

x_train,x_test,y_train,y_test = train_test_split(x, y, test_size = 0.2, random_state = 42)

In [28]:
model = Sequential()
model.add(Embedding(total_words, 100, input_length = max_seq_length-1))
model.add(Bidirectional(LSTM(150)))
model.add(Dense(total_words, activation='softmax'))
model.compile(loss = 'categorical_crossentropy', optimizer = 'adam', metrics = ['accuracy'])

In [29]:
history = model.fit(x_train, y_train, epochs = 50, verbose = 1)

Epoch 1/50
644/644 ━━━━━━━━━━━━━━━━━━━━ 88s 123ms/step - accuracy: 0.0309 - loss: 7.0994
Epoch 2/50
644/644 ━━━━━━━━━━━━━━━━━━━━ 61s 94ms/step - accuracy: 0.0454 - loss: 6.3553
Epoch 3/50
644/644 ━━━━━━━━━━━━━━━━━━━━ 64s 100ms/step - accuracy: 0.0590 - loss: 6.1174
Epoch 4/50
644/644 ━━━━━━━━━━━━━━━━━━━━ 78s 94ms/step - accuracy: 0.0776 - loss: 5.7419
Epoch 5/50
644/644 ━━━━━━━━━━━━━━━━━━━━ 63s 98ms/step - accuracy: 0.0936 - loss: 5.4100
Epoch 6/50
644/644 ━━━━━━━━━━━━━━━━━━━━ 80s 94ms/step - accuracy: 0.1112 - loss: 5.0492
Epoch 7/50
644/644 ━━━━━━━━━━━━━━━━━━━━ 62s 96ms/step - accuracy: 0.1359 - loss: 4.6731
Epoch 8/50
644/644 ━━━━━━━━━━━━━━━━━━━━ 66s 103ms/step - accuracy: 0.1713 - loss: 4.3310
Epoch 9/50
644/644 ━━━━━━━━━━━━━━━━━━━━ 65s 101ms/step - accuracy: 0.2177 - loss: 4.0270
Epoch 10/50
644/644 ━━━━━━━━━━━━━━━━━━━━ 60s 93ms/step - accuracy: 0.2737 - loss: 3.6891
Epoch 11/50
644/644 ━━━━━━━━━━━━━━━━━━━━ 60s 93ms/step - accuracy: 0.3263 - loss: 3.4110
Epoch 12/50
644/644 ━━━━━━

In [31]:
def predict_next_word(model, tokenizer, text, max_sequence_length):
  token_list = tokenizer.texts_to_sequences([text])[0]

  if len(token_list) >= max_sequence_length:
    token_list = token_list[-(max_sequence_length):]

  token_list = pad_sequences([token_list], maxlen=max_sequence_length-1, padding='pre')
  predicted = model.predict(token_list,verbose = 0)
  predicted_word_index = np.argmax(predicted, axis = 1)

  for word, index in tokenizer.word_index.items():
    if index == predicted_word_index:
      return word
  return None

In [32]:
input_text = input("Enter Text: ")
print(f"Input text: {input_text}")
max_sequence_length = model.input_shape[1] + 1
next_word = predict_next_word(model, tokenizer, input_text, max_sequence_length)
print(f"Next word: {next_word}")

Enter Text: Taken to
Input text: Taken to
Next word: wife


In [33]:
y_pred = model.predict(x_test)
rmse = np.sqrt(np.mean((y_test - y_pred) ** 2))
print("RMSE:", rmse)

161/161 ━━━━━━━━━━━━━━━━━━━━ 4s 21ms/step
RMSE: 0.016214869132690858
